In [1]:
import numpy as np
import pandas as pd

In [2]:
df_clean = pd.read_parquet("data/clean.parquet")

In [8]:
df_clean

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,is_attrition,is_overtime,is_top
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,8,0,1,6,4,0,5,1,1,0
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,10,3,3,10,7,1,7,0,0,1
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,7,3,3,0,0,0,0,1,1,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,8,3,3,8,7,3,0,0,1,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,6,3,3,2,2,2,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,2,Medical,2061,3,...,17,3,3,5,2,0,3,0,0,0
1466,39,No,Travel_Rarely,613,Research & Development,6,1,Medical,2062,4,...,9,5,3,7,7,1,7,0,0,1
1467,27,No,Travel_Rarely,155,Research & Development,4,3,Life Sciences,2064,2,...,6,0,3,6,2,0,3,0,1,1
1468,49,No,Travel_Frequently,1023,Sales,2,3,Medical,2065,4,...,17,3,2,9,6,0,8,0,0,0


In [9]:
pd.crosstab(df_clean["JobLevel"], df_clean["PerformanceRating"])

PerformanceRating,3,4
JobLevel,,
1,455,88
2,454,80
3,185,33
4,88,18
5,62,7


In [10]:
df_clean["is_top"] = ((df_clean["PerformanceRating"] == 4) | (df_clean["JobLevel"] >= 3)).astype(int)

In [6]:
top_df = df_clean[df_clean["is_top"] == 1]
rest_df = df_clean[df_clean["is_top"] == 0]

print(f"Всего сотрудников: {len(df_clean)} | Общий отток: {df_clean["is_attrition"].mean():.1%}")
print(f"Топы: {len(top_df)} чел. | Отток топов: {top_df["is_attrition"].mean():.1%}")
print(f"Остальные: {len(rest_df)} чел. | Отток остальных: {rest_df["is_attrition"].mean():.1%}\n")

Всего сотрудников: 1470 | Общий отток: 16.1%
Топы: 561 чел. | Отток топов: 13.4%
Остальные: 909 чел. | Отток остальных: 17.8%


In [17]:
num_cols = (df_clean.select_dtypes(include=[np.number]).
            columns.drop(["is_attrition", "is_top", "PerformanceRating", "EmployeeNumber"]))
num_cols

Index(['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel',
       'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'PercentSalaryHike', 'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager', 'is_overtime'],
      dtype='str')

In [18]:
df_clean[df_clean["is_top"] == False][num_cols].nunique()

Age                          43
DailyRate                   662
DistanceFromHome             29
Education                     5
EnvironmentSatisfaction       4
HourlyRate                   71
JobInvolvement                4
JobLevel                      2
JobSatisfaction               4
MonthlyIncome               834
MonthlyRate                 897
NumCompaniesWorked           10
PercentSalaryHike             9
RelationshipSatisfaction      4
StockOptionLevel              4
TotalWorkingYears            21
TrainingTimesLastYear         7
WorkLifeBalance               4
YearsAtCompany               21
YearsInCurrentRole           16
YearsSinceLastPromotion      14
YearsWithCurrManager         17
is_overtime                   2
dtype: int64

In [19]:
corr_comparison = pd.DataFrame({
    "Spearman_Top": top_df[num_cols].corrwith(top_df["is_attrition"], method="spearman"),
    "Spearman_Rest": rest_df[num_cols].corrwith(rest_df["is_attrition"], method="spearman")
})

In [20]:
corr_comparison["Top_Rest_Ratio"] = corr_comparison["Spearman_Top"] / corr_comparison["Spearman_Rest"]

In [21]:
corr_comparison.sort_values(by="Top_Rest_Ratio", ascending=False).round(3)

,Spearman_Top,Spearman_Rest,Top_Rest_Ratio
TrainingTimesLastYear,-0.123,-0.016,7.529
DistanceFromHome,0.138,0.047,2.914
HourlyRate,-0.012,-0.005,2.558
MonthlyRate,0.023,0.012,1.875
JobSatisfaction,-0.118,-0.096,1.229
EnvironmentSatisfaction,-0.103,-0.096,1.074
StockOptionLevel,-0.180,-0.168,1.073
JobInvolvement,-0.122,-0.120,1.016
MonthlyIncome,-0.196,-0.202,0.972
JobLevel,-0.181,-0.201,0.903


In [22]:
def calc_lift(data, col):
    base_rate = data["is_attrition"].mean()
    stats = data.groupby(col)["is_attrition"].agg(count="count", attrition_rate="mean")
    stats["lift"] = stats["attrition_rate"] / base_rate
    stats["attrition_rate"] = (stats["attrition_rate"] * 100).round(1)
    stats["lift"] = stats["lift"].round(2)
    return stats

In [23]:
# LIFT ДЛЯ ТОПОВ ПО ПЕРЕРАБОТКАМ
calc_lift(top_df, "OverTime")

,count,attrition_rate,lift
OverTime,,,
No,395,8.9,0.66
Yes,166,24.1,1.80


In [24]:
# LIFT ДЛЯ ОСТАЛЬНЫХ ПО ПЕРЕРАБОТКАМ
calc_lift(rest_df, "OverTime")

,count,attrition_rate,lift
OverTime,,,
No,659,11.4,0.64
Yes,250,34.8,1.95


In [25]:
# LIFT ПО ОПЦИОНАМ (ТОПЫ)
calc_lift(top_df, "StockOptionLevel")

,count,attrition_rate,lift
StockOptionLevel,,,
0,227,22.0,1.65
1,253,6.7,0.50
2,52,5.8,0.43
3,29,17.2,1.29


In [26]:
top_df = top_df.copy()
top_df["median_role_income"] = top_df.groupby(["JobRole", "JobLevel"])["MonthlyIncome"].transform("median")
top_df["compa_ratio"] = top_df["MonthlyIncome"] / top_df["median_role_income"]
top_df["is_below_median"] = (top_df["compa_ratio"] < 1.0).astype(int)

In [27]:
# LIFT ПО ЗАРПЛАТЕ НИЖЕ МЕДИАНЫ РОЛИ (ТОПЫ)
calc_lift(top_df, "is_below_median")

,count,attrition_rate,lift
is_below_median,,,
0,288,12.5,0.94
1,273,14.3,1.07


In [28]:
# LIFT ПО ОБУЧЕНИЮ (ТОПЫ)
calc_lift(top_df, "TrainingTimesLastYear")

,count,attrition_rate,lift
TrainingTimesLastYear,,,
0,20,30.0,2.24
1,27,11.1,0.83
2,221,18.1,1.35
3,181,7.7,0.58
4,43,14.0,1.04
5,46,10.9,0.81
6,23,4.3,0.33


In [29]:
top_df["far_from_home"] = (top_df["DistanceFromHome"] >= 15).astype(int)
print("\n--- LIFT ПО ДАЛЬНЕЙ ДОРОГЕ >= 15 (ТОПЫ) ---")
calc_lift(top_df, "far_from_home")


--- LIFT ПО ДАЛЬНЕЙ ДОРОГЕ >= 15 (ТОПЫ) ---


,count,attrition_rate,lift
far_from_home,,,
0,422,11.4,0.85
1,139,19.4,1.45


In [32]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["MonthlyIncome", "is_overtime", "StockOptionLevel", "TrainingTimesLastYear", "DistanceFromHome"]

tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=42)
tree.fit(top_df[features], top_df["is_attrition"])

print(export_text(tree, feature_names=features))

|--- MonthlyIncome <= 4607.00
|   |--- is_overtime <= 0.50
|   |   |--- StockOptionLevel <= 0.50
|   |   |   |--- class: 0
|   |   |--- StockOptionLevel >  0.50
|   |   |   |--- class: 0
|   |--- is_overtime >  0.50
|   |   |--- class: 1
|--- MonthlyIncome >  4607.00
|   |--- MonthlyIncome <= 10661.50
|   |   |--- MonthlyIncome <= 7304.50
|   |   |   |--- class: 0
|   |   |--- MonthlyIncome >  7304.50
|   |   |   |--- class: 0
|   |--- MonthlyIncome >  10661.50
|   |   |--- is_overtime <= 0.50
|   |   |   |--- class: 0
|   |   |--- is_overtime >  0.50
|   |   |   |--- class: 0


In [33]:
from scipy.stats import spearmanr
from sklearn.tree import DecisionTreeClassifier

In [34]:
exclude_cols = ["is_attrition", "is_top", "EmployeeNumber"]
candidate_features = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c not in exclude_cols]

In [39]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# for column in candidate_features:
#     fig, ax = plt.subplots()
#     sns.kdeplot(data=top_df, x=column, hue="Attrition", common_norm=False, fill=True, ax=ax)
#     ax.set_title("Где кривая ушедших (Yes) выше оставшихся — там зона риска")
#     plt.show()

In [40]:
sig_features = []
for col in candidate_features:
    if top_df[col].nunique() <= 1:
        continue
    _, pval = spearmanr(top_df[col], top_df["is_attrition"])
    if pval < 0.05:
        sig_features.append(col)

In [36]:
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=42)
tree.fit(top_df[sig_features], top_df["is_attrition"])

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",30
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples i

In [ ]:
importances = pd.Series(tree.feature_importances_, index=sig_features)
print("Дерево выбрало признаки:")
importances[importances > 0].sort_values(ascending=False).round(3)

In [ ]:
df_clean["Segment"] = df_clean["is_top"].map({1: "Top Talent", 0: "Core Staff"})
df_clean["risk_proba"] = tree.predict_proba(df_clean[sig_features])[:, 1]

In [ ]:
base_top_rate = top_df["is_attrition"].mean()
df_clean["Risk_Level"] = np.select(
    [
        df_clean["risk_proba"] >= base_top_rate * 2,  # риск в 2+ раза выше среднего по топам
        df_clean["risk_proba"] >= base_top_rate  # риск выше среднего по топам
    ],
    [
        "3. High Risk (Критичный)",
        "2. Medium Risk (Повышенный)"
    ],
    default="1. Low Risk (Стабильный)"
)

In [ ]:
# 4. Проверяем валидацию на Топах
val = df_clean[df_clean["is_top"] == 1].groupby("Risk_Level").agg(
    total_top=("is_attrition", "count"),
    left_company=("is_attrition", "sum"),
    active_lookalikes=("is_attrition", lambda x: (x == 0).sum()),
    attrition_rate=("is_attrition", lambda x: round(x.mean() * 100, 1))
)
print("--- ВАЛИДАЦИЯ НА ТОПАХ ---")
print(val)

In [ ]:
# 5. Выгружаем финальную витрину для Tableau
df_clean.to_csv("avito_hr_datamart.csv", index=False)
print("\nВитрина avito_hr_datamart.csv готова для загрузки в Tableau!")